|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>The block allocator<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: build the allocator and the page table<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
import numpy as np

rng = np.random.default_rng(0)

Build the allocator.

You need no tensors and no GPU. This is stage 06 of the ladder. It is pure
bookkeeping. Get it correct on its own, before a kernel must read through
it.

In [2]:
### run this cell

BLOCK    = 16                # tokens per block
N_BLOCKS = 4096              # blocks in the pool
lengths  = rng.lognormal(mean=np.log(120), sigma=0.9, size=2000).astype(int) + 1

print(f'pool holds {N_BLOCKS*BLOCK:,} tokens in {N_BLOCKS:,} blocks of {BLOCK}')

pool holds 65,536 tokens in 4,096 blocks of 16


# Exercise 1: the free list

You need a pool of blocks, a stack of the free ones, and one reference count
per block. The reference count looks unnecessary now. Exercise 4 explains
it.

In [3]:
class BlockAllocator:
  def __init__(self, n_blocks):
    self.free = list(range(n_blocks))      # a stack of physical ids
    self.ref  = [0] * n_blocks

  def allocate(self):
    if not self.free:
      raise MemoryError('out of KV blocks')
    b = self.free.pop()
    self.ref[b] = 1
    return b

  def release(self, b):
    self.ref[b] -= 1
    if self.ref[b] == 0:
      self.free.append(b)

  def share(self, b):
    self.ref[b] += 1
    return b

  @property
  def used(self):
    return len(self.ref) - len(self.free)

a = BlockAllocator(8)
x, y = a.allocate(), a.allocate()
print('used after 2 allocations:', a.used)
a.release(x)
print('used after 1 release:    ', a.used)

used after 2 allocations: 2
used after 1 release:     1


# Exercise 2: the block table

Each sequence holds one table. The table maps a logical token position onto a
physical slot in the pool. It grows one block at a time, as the sequence
grows.

In [4]:
class BlockTable:
  """One sequence's view of the pool."""
  def __init__(self, allocator, block_size):
    self.alloc  = allocator
    self.bs     = block_size
    self.blocks = []      # logical block index -> physical block id
    self.n      = 0       # tokens held

  def append_token(self):
    if self.n % self.bs == 0:            # the current block is full
      self.blocks.append(self.alloc.allocate())
    self.n += 1

  def slot_index(self, pos):
    """logical token position -> flat slot in the pool"""
    return self.blocks[pos // self.bs] * self.bs + pos % self.bs

  def free(self):
    for b in self.blocks:
      self.alloc.release(b)
    self.blocks, self.n = [], 0

alloc = BlockAllocator(N_BLOCKS)
t = BlockTable(alloc, BLOCK)
for _ in range(40):
  t.append_token()
print(f'40 tokens -> {len(t.blocks)} blocks: {t.blocks}')
print(f'position 0  -> slot {t.slot_index(0)}')
print(f'position 17 -> slot {t.slot_index(17)}')

40 tokens -> 3 blocks: [4095, 4094, 4093]
position 0  -> slot 65520
position 17 -> slot 65505


# Exercise 3: how many sequences fit now?

Admit requests from the workload until the allocator refuses. Then compare
your answer against a reservation of `max_len` for each request.

In [5]:
alloc  = BlockAllocator(N_BLOCKS)
tables = []
admitted = 0

for L in lengths:
  t = BlockTable(alloc, BLOCK)
  try:
    for _ in range(int(L)):
      t.append_token()
  except MemoryError:
    t.free()
    break
  tables.append(t); admitted += 1

held  = sum(len(t.blocks) for t in tables) * BLOCK
used  = sum(t.n for t in tables)
print(f'admitted {admitted} sequences before the pool ran out')
print(f'tokens held {held:,}, tokens used {used:,}  -> {100*(1-used/held):.1f}% wasted')

MAX_LEN = 2048
contig  = (N_BLOCKS*BLOCK) // MAX_LEN
print(f'\ncontiguous, reserving {MAX_LEN}: {contig} sequences')
print(f'paged:                     {admitted} sequences   ({admitted/contig:.0f}x)')

admitted 358 sequences before the pool ran out
tokens held 65,376, tokens used 62,760  -> 4.0% wasted

contiguous, reserving 2048: 32 sequences
paged:                     358 sequences   (11x)


# Exercise 4: two sequences, one prompt

Parallel sampling asks the model for four replies to one prompt. All four
replies share the same K and V for the prompt.

A page table makes that free.

In [6]:
alloc = BlockAllocator(N_BLOCKS)
before = alloc.used

parent = BlockTable(alloc, BLOCK)
for _ in range(64):
  parent.append_token()
after_parent = alloc.used

# four samples from the same prompt: share every block the prompt holds
children = []
for _ in range(4):
  c = BlockTable(alloc, BLOCK)
  c.blocks = [alloc.share(b) for b in parent.blocks]
  c.n = parent.n
  children.append(c)

print(f'prompt of 64 tokens costs      {after_parent - before} blocks')
print(f'4 samples sharing it cost      {alloc.used - after_parent} more')
print(f'4 samples copying it would be  {4*(after_parent-before)} more')

for c in children:
  c.free()
print(f'\nafter the children leave, still held: {alloc.used} blocks (the parent)')

prompt of 64 tokens costs      4 blocks
4 samples sharing it cost      0 more
4 samples copying it would be  16 more

after the children leave, still held: 4 blocks (the parent)


### What you built

You built an allocator, a page table, and a reference count. That is about
sixty lines and no tensors. It gives about ten times the memory efficiency of
the code it replaces.

Remember Exercise 4. A shared prefix became **a pointer operation**. Four
samples from one prompt cost four block-table entries, not four copies of the
prompt's KV cache. The reference count also stops one sequence from freeing a
block that another sequence still reads.

Follow that idea and you reach the rest of stage 09:

- One sequence writes into a shared block. Copy the block first, for that
  sequence only. This is copy-on-write, exactly as `fork` uses it.
- Hash the contents of a block. Two unrelated **requests** with the same
  system prompt then share it. This is automatic prefix caching, which is a
  page cache.

A contiguous buffer offered none of this. Paging did not make sharing faster.
Paging made sharing possible to express.

    ./vc guide 6